In [1]:
# ============================================================================
# MSGR-RAG Component 2: Entity Extraction
# ============================================================================
# Purpose: Extract relationship triples from DrugBank for entity-level graph
# Author: Pavan Kumar Reddy Yannam
# Date: December 2024
# ============================================================================

In [2]:
# # Component 2: Entity Extraction
# 
# **Objectives:**
# 1. Parse `drug-interactions` field → Extract (Drug_A, interacts_with, Drug_B)
# 2. Parse `toxicity` field → Extract (Drug, causes, Adverse_Effect)
# 3. Parse `description`/`indication` → Extract (Drug, treats, Condition)
# 4. Build entity catalog for G_E
# 5. Create relationship edge list for entity-level graph

In [3]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [4]:
# Define paths
DATA_DIR = Path("../Datasets")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
DRUGBANK_PATH = DATA_DIR / "drugbank_clean.csv"
SELECTED_DRUGS_PATH = INTERMEDIATE_DIR / "selected_drugs_100.csv"

# Output paths for this component
ENTITY_CATALOG_PATH = INTERMEDIATE_DIR / "entity_catalog.csv"
ENTITY_RELATIONSHIPS_PATH = INTERMEDIATE_DIR / "entity_relationships.csv"
EXTRACTION_REPORT_PATH = INTERMEDIATE_DIR / "entity_extraction_report.json"

print(f"✓ Input: {SELECTED_DRUGS_PATH}")
print(f"✓ Outputs:")
print(f"  - {ENTITY_CATALOG_PATH}")
print(f"  - {ENTITY_RELATIONSHIPS_PATH}")
print(f"  - {EXTRACTION_REPORT_PATH}")

✓ Input: ../Datasets/intermediate/selected_drugs_100.csv
✓ Outputs:
  - ../Datasets/intermediate/entity_catalog.csv
  - ../Datasets/intermediate/entity_relationships.csv
  - ../Datasets/intermediate/entity_extraction_report.json


In [5]:
# ## Section 2: Load Data

In [6]:
print("Loading data...")
print("=" * 80)

# Load full DrugBank dataset
drugbank_df = pd.read_csv(DRUGBANK_PATH, low_memory=False)
drugbank_df = drugbank_df.dropna(how='all')

# Load selected 100 drugs
selected_df = pd.read_csv(SELECTED_DRUGS_PATH)

print(f"✓ Full DrugBank: {len(drugbank_df):,} drugs")
print(f"✓ Selected drugs: {len(selected_df)} drugs")

# Merge to get full data for selected drugs
selected_ids = selected_df['drugbank-id'].tolist()
working_df = drugbank_df[drugbank_df['drugbank-id'].isin(selected_ids)].copy()

print(f"✓ Working dataset: {len(working_df)} drugs")
print(f"\nColumns available: {len(working_df.columns)}")

Loading data...
✓ Full DrugBank: 16,761 drugs
✓ Selected drugs: 100 drugs
✓ Working dataset: 120 drugs

Columns available: 40


In [7]:
# ## Section 3: Examine Data Formats
# 
# Before extraction, let's understand the actual format of each field

In [8]:
print("Examining Data Formats:")
print("=" * 80)

# ============================================================================
# 3.1: Drug-Interactions Field Format
# ============================================================================
print("\n1. DRUG-INTERACTIONS FIELD FORMAT")
print("-" * 80)

if 'drug-interactions' in working_df.columns:
    sample_interactions = working_df['drug-interactions'].dropna().head(3)
    
    for i, interaction in enumerate(sample_interactions, 1):
        print(f"\nSample {i}:")
        print(f"Type: {type(interaction)}")
        print(f"Length: {len(str(interaction))} characters")
        print(f"Content preview (first 500 chars):")
        print(str(interaction)[:500])
        print("\n" + "-" * 40)
    
    # Detect delimiter
    if len(sample_interactions) > 0:
        first_sample = str(sample_interactions.iloc[0])
        
        # Check for common delimiters
        delimiters = {
            '|': first_sample.count('|'),
            ';': first_sample.count(';'),
            '\n': first_sample.count('\n'),
            ',': first_sample.count(',')
        }
        
        print("\nDelimiter detection:")
        for delim, count in delimiters.items():
            if count > 0:
                print(f"  '{delim}': {count} occurrences")
        
        likely_delimiter = max(delimiters, key=delimiters.get) if max(delimiters.values()) > 0 else None
        print(f"\nLikely delimiter: '{likely_delimiter}' (most frequent)")
else:
    print("✗ drug-interactions column not found")

# ============================================================================
# 3.2: Toxicity Field Format
# ============================================================================
print("\n\n2. TOXICITY FIELD FORMAT")
print("-" * 80)

if 'toxicity' in working_df.columns:
    sample_toxicity = working_df['toxicity'].dropna().head(3)
    
    for i, tox in enumerate(sample_toxicity, 1):
        print(f"\nSample {i}:")
        print(f"Length: {len(str(tox))} characters")
        print(f"Content preview (first 500 chars):")
        print(str(tox)[:500])
        print("\n" + "-" * 40)
else:
    print("✗ toxicity column not found")

# ============================================================================
# 3.3: Description/Indication Field Format
# ============================================================================
print("\n\n3. DESCRIPTION/INDICATION FIELD FORMAT")
print("-" * 80)

for field in ['description', 'indication']:
    if field in working_df.columns:
        print(f"\n{field.upper()}:")
        sample = working_df[field].dropna().head(2)
        
        for i, text in enumerate(sample, 1):
            print(f"\nSample {i}:")
            print(f"Length: {len(str(text))} characters")
            print(f"Content preview (first 300 chars):")
            print(str(text)[:300])
            print("\n" + "-" * 40)
    else:
        print(f"\n{field.upper()}: Not found")

Examining Data Formats:

1. DRUG-INTERACTIONS FIELD FORMAT
--------------------------------------------------------------------------------

Sample 1:
Type: <class 'str'>
Length: 5215 characters
Content preview (first 500 chars):
DB13821 DB00328 DB08953 DB00682 DB00928 DB09124 DB13400 DB09381 DB00656 DB09052 DB06679 DB06737 DB03374 DB08814 DB12289 DB06251 DB09217 DB11133 DB00970 DB01066 DB13875 DB13876 DB00034 DB00814 DB00562 DB00036 DB09253 DB01247 DB14006 DB01942 DB00642 DB13510 DB13286 DB00495 DB00232 DB13432 DB13199 DB00105 DB06228 DB13947 DB00305 DB00605 DB00957 DB06715 DB06772 DB01327 DB01543 DB08984 DB01229 DB12235 DB00583 DB07615 DB14473 DB00100 DB13912 DB01140 DB01590 DB11419 DB04066 DB09030 DB00806 DB03619 DB11

----------------------------------------

Sample 2:
Type: <class 'str'>
Length: 5615 characters
Content preview (first 500 chars):
DB13821 DB00328 DB00682 DB08953 DB00928 DB09124 DB13400 DB09381 DB06196 DB00656 DB09052 DB06679 DB06737 DB03374 DB00178 DB08814 DB12289 D

In [9]:
# ## Section 4: Extract 'interacts_with' Relationships
# 
# Parse drug-interactions field to extract (Drug_A, interacts_with, Drug_B) triples


In [25]:
# Diagnostic: Check what's actually in drug-interactions
print("DIAGNOSTIC: Examining drug-interactions field")
print("=" * 80)

sample = working_df[working_df['drug-interactions'].notna()].head(1)

if len(sample) > 0:
    print(f"\nDrug name: {sample.iloc[0]['name']}")
    print(f"\nFull drug-interactions content:")
    print(sample.iloc[0]['drug-interactions'])
    print("\n" + "=" * 80)
else:
    print("No drugs with interaction data found!")

DIAGNOSTIC: Examining drug-interactions field

Drug name: Lepirudin

Full drug-interactions content:
DB13821 DB00328 DB08953 DB00682 DB00928 DB09124 DB13400 DB09381 DB00656 DB09052 DB06679 DB06737 DB03374 DB08814 DB12289 DB06251 DB09217 DB11133 DB00970 DB01066 DB13875 DB13876 DB00034 DB00814 DB00562 DB00036 DB09253 DB01247 DB14006 DB01942 DB00642 DB13510 DB13286 DB00495 DB00232 DB13432 DB13199 DB00105 DB06228 DB13947 DB00305 DB00605 DB00957 DB06715 DB06772 DB01327 DB01543 DB08984 DB01229 DB12235 DB00583 DB07615 DB14473 DB00100 DB13912 DB01140 DB01590 DB11419 DB04066 DB09030 DB00806 DB03619 DB11330 DB00531 DB12814 DB00535 DB00631 DB00888 DB13136 DB00853 DB09050 DB11201 DB13266 DB00278 DB00367 DB01381 DB15119 DB00851 DB04017 DB00389 DB03404 DB13981 DB00834 DB00773 DB13943 DB04828 DB09062 DB14655 DB09074 DB00008 DB00120 DB01051 DB04818 DB13201 DB11367 DB02659 DB00945 DB00428 DB13563 DB13954 DB00274 DB01033 DB15861 DB12610 DB01240 DB00987 DB12726 DB09249 DB13148 DB00276 DB09084 DB08550 DB0

In [31]:
print("DIAGNOSTIC: unique_interactions structure")
print(f"Type: {type(unique_interactions)}")
print(f"Length: {len(unique_interactions)}")
if len(unique_interactions) > 0:
    print(f"First element type: {type(unique_interactions[0])}")
    print(f"First 5 elements:")
    for i, item in enumerate(unique_interactions[:5]):
        print(f"  {i}: {item}")

DIAGNOSTIC: unique_interactions structure
Type: <class 'list'>
Length: 103219
First element type: <class 'tuple'>
First 5 elements:
  0: ('Lepirudin', 'interacts_with', 'Ceftezole')
  1: ('Lepirudin', 'interacts_with', 'Indomethacin')
  2: ('Lepirudin', 'interacts_with', 'Indalpine')
  3: ('Lepirudin', 'interacts_with', 'Warfarin')
  4: ('Lepirudin', 'interacts_with', 'Azacitidine')


In [26]:
print("Extracting 'interacts_with' Relationships:")
print("=" * 80)

def extract_drug_interactions(row):
    """
    Extract drug-drug interaction triples from drug-interactions field
    
    Returns: List of (subject_drug, predicate, object_drug) tuples
    """
    interactions = []
    
    if pd.isna(row['drug-interactions']):
        return interactions
    
    subject_drug = row['name']  # The drug in this row
    subject_id = row['drugbank-id']  # DrugBank ID of this drug
    interaction_text = str(row['drug-interactions'])
    
    # The field contains space-separated DrugBank IDs (e.g., "DB13821 DB00328 DB08953")
    # Extract all DrugBank IDs
    drugbank_id_pattern = r'DB\d{5}'
    interacting_ids = re.findall(drugbank_id_pattern, interaction_text)
    
    # Look up drug names for these IDs
    for db_id in interacting_ids:
        # Find the drug name for this DrugBank ID
        matching_drug = drugbank_df[drugbank_df['drugbank-id'] == db_id]
        
        if len(matching_drug) > 0:
            object_drug = matching_drug.iloc[0]['name']
            
            # Avoid self-interactions
            if db_id != subject_id:
                interactions.append((subject_drug, 'interacts_with', object_drug))
    
    return interactions

# Apply extraction
print("Processing drug interactions...")
all_interactions = []

for idx, row in working_df.iterrows():
    interactions = extract_drug_interactions(row)
    all_interactions.extend(interactions)

print(f"✓ Extracted {len(all_interactions)} raw interaction triples")

# Remove duplicates (same pair in both directions counts once)
unique_interactions = []
seen_pairs = set()

for subj, pred, obj in all_interactions:
    # Create canonical pair (alphabetically sorted)
    pair = tuple(sorted([subj.lower(), obj.lower()]))
    
    if pair not in seen_pairs:
        seen_pairs.add(pair)
        unique_interactions.append((subj, pred, obj))

print(f"✓ Unique interactions after deduplication: {len(unique_interactions)}")

# Sample output
print("\nSample interactions:")
for i, (s, p, o) in enumerate(unique_interactions[:10], 1):
    print(f"{i:2d}. ({s}, {p}, {o})")

# %%
# Statistics
print("\n" + "=" * 80)
print("INTERACTS_WITH EXTRACTION STATISTICS")
print("=" * 80)

# Count interactions per drug
interaction_counts = Counter([triple[0] for triple in unique_interactions])

print(f"\nTotal unique interactions: {len(unique_interactions)}")
print(f"Drugs with interactions: {len(interaction_counts)}")

# Fixed division by zero
if len(interaction_counts) > 0:
    print(f"Average interactions per drug: {len(unique_interactions) / len(interaction_counts):.1f}")
else:
    print(f"Average interactions per drug: 0")

print(f"\nTop 10 drugs by interaction count:")
for drug, count in interaction_counts.most_common(10):
    print(f"  {drug:30s}: {count} interactions")

Extracting 'interacts_with' Relationships:
Processing drug interactions...
✓ Extracted 105169 raw interaction triples
✓ Unique interactions after deduplication: 103219

Sample interactions:
 1. (Lepirudin, interacts_with, Ceftezole)
 2. (Lepirudin, interacts_with, Indomethacin)
 3. (Lepirudin, interacts_with, Indalpine)
 4. (Lepirudin, interacts_with, Warfarin)
 5. (Lepirudin, interacts_with, Azacitidine)
 6. (Lepirudin, interacts_with, Medrogestone)
 7. (Lepirudin, interacts_with, Linsidomine)
 8. (Lepirudin, interacts_with, Esterified estrogens)
 9. (Lepirudin, interacts_with, Trazodone)
10. (Lepirudin, interacts_with, Blinatumomab)

INTERACTS_WITH EXTRACTION STATISTICS

Total unique interactions: 103219
Drugs with interactions: 100
Average interactions per drug: 1032.2

Top 10 drugs by interaction count:
  Zonisamide                    : 2249 interactions
  Stiripentol                   : 2100 interactions
  Promethazine                  : 2004 interactions
  Acetylsalicylic acid   

In [11]:
# ## Section 5: Extract 'causes' Relationships
# 
# Parse toxicity field to extract (Drug, causes, Adverse_Effect) triples


In [13]:
print("Extracting 'causes' Relationships:")
print("=" * 80)

# Define toxicity keywords and adverse effects
TOXICITY_PATTERNS = {
    'hepatotoxic': 'Hepatotoxicity',
    'hepatotoxicity': 'Hepatotoxicity',
    'liver toxicity': 'Hepatotoxicity',
    'liver damage': 'Hepatotoxicity',
    
    'nephrotoxic': 'Nephrotoxicity',
    'nephrotoxicity': 'Nephrotoxicity',
    'kidney toxicity': 'Nephrotoxicity',
    'kidney damage': 'Nephrotoxicity',
    'renal toxicity': 'Nephrotoxicity',
    
    'cardiotoxic': 'Cardiotoxicity',
    'cardiotoxicity': 'Cardiotoxicity',
    'cardiac toxicity': 'Cardiotoxicity',
    'heart toxicity': 'Cardiotoxicity',
    
    'neurotoxic': 'Neurotoxicity',
    'neurotoxicity': 'Neurotoxicity',
    'neurological toxicity': 'Neurotoxicity',
    
    'carcinogen': 'Carcinogenicity',
    'carcinogenic': 'Carcinogenicity',
    'cancer': 'Carcinogenicity',
    
    'teratogenic': 'Teratogenicity',
    'teratogenicity': 'Teratogenicity',
    'birth defects': 'Teratogenicity',
    
    'ototoxic': 'Ototoxicity',
    'ototoxicity': 'Ototoxicity',
    'hearing loss': 'Ototoxicity',
    
    'bleeding': 'Bleeding',
    'hemorrhage': 'Bleeding',
    
    'nausea': 'Nausea',
    'vomiting': 'Vomiting',
    'diarrhea': 'Diarrhea',
    'constipation': 'Constipation',
    
    'drowsiness': 'Drowsiness',
    'sedation': 'Sedation',
    'dizziness': 'Dizziness',
    
    'hypotension': 'Hypotension',
    'hypertension': 'Hypertension',
    
    'hyperglycemia': 'Hyperglycemia',
    'hypoglycemia': 'Hypoglycemia',
}

def extract_toxicity_effects(row):
    """
    Extract adverse effects from toxicity field
    
    Returns: List of (drug, predicate, adverse_effect) tuples
    """
    effects = []
    
    if pd.isna(row['toxicity']):
        return effects
    
    drug = row['name']
    toxicity_text = str(row['toxicity']).lower()
    
    # Check for each toxicity pattern
    for pattern, effect_name in TOXICITY_PATTERNS.items():
        if pattern in toxicity_text:
            effects.append((drug, 'causes', effect_name))
    
    return effects

# Apply extraction
print("Processing toxicity data...")
all_toxicity = []

for idx, row in working_df.iterrows():
    effects = extract_toxicity_effects(row)
    all_toxicity.extend(effects)

print(f"✓ Extracted {len(all_toxicity)} raw toxicity triples")

# Remove duplicates
unique_toxicity = list(set(all_toxicity))
print(f"✓ Unique toxicity relationships: {len(unique_toxicity)}")

# Sample output
print("\nSample toxicity relationships:")
for i, (s, p, o) in enumerate(unique_toxicity[:15], 1):
    print(f"{i:2d}. ({s}, {p}, {o})")

# %%
# Statistics
print("\n" + "=" * 80)
print("CAUSES EXTRACTION STATISTICS")
print("=" * 80)

# Count by drug
toxicity_by_drug = Counter([triple[0] for triple in unique_toxicity])

# Count by effect
toxicity_by_effect = Counter([triple[2] for triple in unique_toxicity])

print(f"\nTotal unique toxicity relationships: {len(unique_toxicity)}")
print(f"Drugs with toxicity data: {len(toxicity_by_drug)}")
print(f"Unique adverse effects identified: {len(toxicity_by_effect)}")
print(f"Average effects per drug: {len(unique_toxicity) / len(toxicity_by_drug):.1f}")

print(f"\nTop 10 drugs by toxicity relationships:")
for drug, count in toxicity_by_drug.most_common(10):
    print(f"  {drug:30s}: {count} effects")

print(f"\nMost common adverse effects:")
for effect, count in toxicity_by_effect.most_common(15):
    print(f"  {effect:30s}: {count} drugs")


Extracting 'causes' Relationships:
Processing toxicity data...
✓ Extracted 155 raw toxicity triples
✓ Unique toxicity relationships: 144

Sample toxicity relationships:
 1. (Midodrine, causes, Hypertension)
 2. (Ifosfamide, causes, Nausea)
 3. (Rifampicin, causes, Vomiting)
 4. (Tiagabine, causes, Drowsiness)
 5. (Ranitidine, causes, Hypotension)
 6. (Flupentixol, causes, Hypotension)
 7. (Drospirenone, causes, Nausea)
 8. (Telithromycin, causes, Dizziness)
 9. (Romidepsin, causes, Nausea)
10. (Alfacalcidol, causes, Constipation)
11. (Thiothixene, causes, Dizziness)
12. (Ganciclovir, causes, Carcinogenicity)
13. (Misoprostol, causes, Diarrhea)
14. (Iloprost, causes, Hypotension)
15. (Tiagabine, causes, Vomiting)

CAUSES EXTRACTION STATISTICS

Total unique toxicity relationships: 144
Drugs with toxicity data: 67
Unique adverse effects identified: 17
Average effects per drug: 2.1

Top 10 drugs by toxicity relationships:
  Methyldopa                    : 7 effects
  Alfacalcidol          

In [14]:
# ## Section 6: Extract 'treats' Relationships
# 
# Parse description/indication to extract (Drug, treats, Condition) triples


In [15]:
print("Extracting 'treats' Relationships:")
print("=" * 80)

# Define therapeutic patterns and conditions
THERAPEUTIC_PATTERNS = {
    # Cardiovascular
    'hypertension': 'Hypertension',
    'high blood pressure': 'Hypertension',
    'heart failure': 'Heart_Failure',
    'cardiac': 'Cardiac_Conditions',
    'arrhythmia': 'Arrhythmia',
    'angina': 'Angina',
    'myocardial infarction': 'Myocardial_Infarction',
    'stroke': 'Stroke',
    'thrombosis': 'Thrombosis',
    
    # Endocrine
    'diabetes': 'Diabetes',
    'hyperglycemia': 'Hyperglycemia',
    'hypoglycemia': 'Hypoglycemia',
    'thyroid': 'Thyroid_Disorders',
    'hypothyroidism': 'Hypothyroidism',
    'hyperthyroidism': 'Hyperthyroidism',
    
    # Pain/Inflammation
    'pain': 'Pain',
    'analgesic': 'Pain',
    'inflammation': 'Inflammation',
    'anti-inflammatory': 'Inflammation',
    'arthritis': 'Arthritis',
    'rheumatoid': 'Rheumatoid_Arthritis',
    'fever': 'Fever',
    'antipyretic': 'Fever',
    
    # Infection
    'bacterial': 'Bacterial_Infection',
    'antibiotic': 'Bacterial_Infection',
    'infection': 'Infection',
    'viral': 'Viral_Infection',
    'antiviral': 'Viral_Infection',
    'fungal': 'Fungal_Infection',
    'antifungal': 'Fungal_Infection',
    
    # Neurological
    'seizure': 'Seizure',
    'epilepsy': 'Epilepsy',
    'anticonvulsant': 'Seizure',
    'parkinsons': 'Parkinsons_Disease',
    'alzheimer': 'Alzheimers_Disease',
    'migraine': 'Migraine',
    'headache': 'Headache',
    
    # Psychiatric
    'depression': 'Depression',
    'antidepressant': 'Depression',
    'anxiety': 'Anxiety',
    'anxiolytic': 'Anxiety',
    'psychosis': 'Psychosis',
    'antipsychotic': 'Psychosis',
    'schizophrenia': 'Schizophrenia',
    
    # Respiratory
    'asthma': 'Asthma',
    'bronchodilator': 'Bronchospasm',
    'copd': 'COPD',
    'cough': 'Cough',
    
    # Gastrointestinal
    'ulcer': 'Ulcer',
    'gastric': 'Gastric_Disorders',
    'nausea': 'Nausea',
    'antiemetic': 'Nausea',
    'diarrhea': 'Diarrhea',
    'constipation': 'Constipation',
    
    # Cancer
    'cancer': 'Cancer',
    'tumor': 'Tumor',
    'neoplasm': 'Neoplasm',
    'chemotherapy': 'Cancer',
    'leukemia': 'Leukemia',
    'lymphoma': 'Lymphoma',
}

def extract_therapeutic_indications(row):
    """
    Extract therapeutic indications from description/indication fields
    
    Returns: List of (drug, predicate, condition) tuples
    """
    indications = []
    
    drug = row['name']
    
    # Combine description and indication fields
    text_fields = []
    for field in ['description', 'indication']:
        if field in row.index and pd.notna(row[field]):
            text_fields.append(str(row[field]).lower())
    
    combined_text = ' '.join(text_fields)
    
    if not combined_text:
        return indications
    
    # Check for each therapeutic pattern
    for pattern, condition_name in THERAPEUTIC_PATTERNS.items():
        if pattern in combined_text:
            indications.append((drug, 'treats', condition_name))
    
    return indications

# Apply extraction
print("Processing therapeutic indications...")
all_treats = []

for idx, row in working_df.iterrows():
    indications = extract_therapeutic_indications(row)
    all_treats.extend(indications)

print(f"✓ Extracted {len(all_treats)} raw therapeutic triples")

# Remove duplicates
unique_treats = list(set(all_treats))
print(f"✓ Unique therapeutic relationships: {len(unique_treats)}")

# Sample output
print("\nSample therapeutic relationships:")
for i, (s, p, o) in enumerate(unique_treats[:15], 1):
    print(f"{i:2d}. ({s}, {p}, {o})")

# %%
# Statistics
print("\n" + "=" * 80)
print("TREATS EXTRACTION STATISTICS")
print("=" * 80)

# Count by drug
treats_by_drug = Counter([triple[0] for triple in unique_treats])

# Count by condition
treats_by_condition = Counter([triple[2] for triple in unique_treats])

print(f"\nTotal unique therapeutic relationships: {len(unique_treats)}")
print(f"Drugs with therapeutic data: {len(treats_by_drug)}")
print(f"Unique conditions identified: {len(treats_by_condition)}")
print(f"Average indications per drug: {len(unique_treats) / len(treats_by_drug):.1f}")

print(f"\nTop 10 drugs by therapeutic relationships:")
for drug, count in treats_by_drug.most_common(10):
    print(f"  {drug:30s}: {count} conditions")

print(f"\nMost common conditions treated:")
for condition, count in treats_by_condition.most_common(15):
    print(f"  {condition:30s}: {count} drugs")

Extracting 'treats' Relationships:
Processing therapeutic indications...
✓ Extracted 213 raw therapeutic triples
✓ Unique therapeutic relationships: 189

Sample therapeutic relationships:
 1. (Flupentixol, treats, Psychosis)
 2. (Canakinumab, treats, Arthritis)
 3. (Ketotifen, treats, Asthma)
 4. (Cromoglicic acid, treats, Asthma)
 5. (Flupentixol, treats, Anxiety)
 6. (Phenylbutazone, treats, Pain)
 7. (Oxaprozin, treats, Pain)
 8. (Etacrynic acid, treats, Hypertension)
 9. (Ranitidine, treats, Ulcer)
10. (Ranitidine, treats, Gastric_Disorders)
11. (Acetylsalicylic acid, treats, Headache)
12. (Felbamate, treats, Epilepsy)
13. (Cariprazine, treats, Psychosis)
14. (Trifluoperazine, treats, Psychosis)
15. (Cefprozil, treats, Infection)

TREATS EXTRACTION STATISTICS

Total unique therapeutic relationships: 189
Drugs with therapeutic data: 83
Unique conditions identified: 42
Average indications per drug: 2.3

Top 10 drugs by therapeutic relationships:
  Acetylsalicylic acid          : 11 c

In [16]:
# ## Section 7: Build Entity Catalog
# 
# Create comprehensive list of all entities (drugs, conditions, effects)


In [17]:
print("Building Entity Catalog:")
print("=" * 80)

# Collect all entities
entities = {}

# 1. Drugs (subjects from all relationships)
print("\n1. Collecting drug entities...")
drug_entities = set()

# From interactions
drug_entities.update([t[0] for t in unique_interactions])
drug_entities.update([t[2] for t in unique_interactions])

# From toxicity
drug_entities.update([t[0] for t in unique_toxicity])

# From treats
drug_entities.update([t[0] for t in unique_treats])

print(f"   ✓ Found {len(drug_entities)} unique drug entities")

# Add to catalog
for drug in drug_entities:
    entities[drug] = {
        'entity_id': f"DRUG_{len(entities)}",
        'entity_name': drug,
        'entity_type': 'Drug',
        'source': 'DrugBank'
    }

# 2. Adverse Effects (objects from toxicity relationships)
print("\n2. Collecting adverse effect entities...")
effect_entities = set([t[2] for t in unique_toxicity])
print(f"   ✓ Found {len(effect_entities)} unique adverse effects")

for effect in effect_entities:
    entities[effect] = {
        'entity_id': f"EFFECT_{len(entities)}",
        'entity_name': effect,
        'entity_type': 'Adverse_Effect',
        'source': 'Toxicity_Extraction'
    }

# 3. Conditions (objects from treats relationships)
print("\n3. Collecting condition entities...")
condition_entities = set([t[2] for t in unique_treats])
print(f"   ✓ Found {len(condition_entities)} unique conditions")

for condition in condition_entities:
    entities[condition] = {
        'entity_id': f"CONDITION_{len(entities)}",
        'entity_name': condition,
        'entity_type': 'Condition',
        'source': 'Therapeutic_Extraction'
    }

print(f"\n" + "=" * 80)
print(f"TOTAL ENTITIES: {len(entities)}")
print(f"  - Drugs: {len(drug_entities)}")
print(f"  - Adverse Effects: {len(effect_entities)}")
print(f"  - Conditions: {len(condition_entities)}")
print("=" * 80)

# %%
# Convert to DataFrame
entity_catalog_df = pd.DataFrame(entities.values())
entity_catalog_df = entity_catalog_df.sort_values('entity_id')

print("\nEntity Catalog Preview:")
print(entity_catalog_df.head(20))

print(f"\nEntity type distribution:")
print(entity_catalog_df['entity_type'].value_counts())

Building Entity Catalog:

1. Collecting drug entities...
   ✓ Found 95 unique drug entities

2. Collecting adverse effect entities...
   ✓ Found 17 unique adverse effects

3. Collecting condition entities...
   ✓ Found 42 unique conditions

TOTAL ENTITIES: 149
  - Drugs: 95
  - Adverse Effects: 17
  - Conditions: 42

Entity Catalog Preview:
         entity_id            entity_name entity_type                  source
112  CONDITION_112                   COPD   Condition  Therapeutic_Extraction
113  CONDITION_113              Infection   Condition  Therapeutic_Extraction
114  CONDITION_114                  Cough   Condition  Therapeutic_Extraction
115  CONDITION_115                 Angina   Condition  Therapeutic_Extraction
116  CONDITION_116         Hypothyroidism   Condition  Therapeutic_Extraction
117  CONDITION_117               Lymphoma   Condition  Therapeutic_Extraction
118  CONDITION_118      Gastric_Disorders   Condition  Therapeutic_Extraction
119  CONDITION_119             Th

In [18]:
# ## Section 8: Create Relationship Edge List

In [32]:
print("Creating Relationship Edge List:")
print("=" * 80)

# Combine all relationships
all_relationships = []

# Add interacts_with relationships
for subj, pred, obj in unique_interactions:
    all_relationships.append({
        'subject': subj,
        'predicate': pred,
        'object': obj,
        'relationship_type': 'Drug-Drug_Interaction',
        'confidence': 1.0,  # Placeholder for now
        'source': 'drug-interactions_field'
    })

# Add causes relationships
for subj, pred, obj in unique_toxicity:
    all_relationships.append({
        'subject': subj,
        'predicate': pred,
        'object': obj,
        'relationship_type': 'Drug-Adverse_Effect',
        'confidence': 1.0,
        'source': 'toxicity_field'
    })

# Add treats relationships
for subj, pred, obj in unique_treats:
    all_relationships.append({
        'subject': subj,
        'predicate': pred,
        'object': obj,
        'relationship_type': 'Drug-Condition',
        'confidence': 1.0,
        'source': 'description_indication_fields'
    })

# Convert to DataFrame
relationships_df = pd.DataFrame(all_relationships)

print(f"✓ Total relationships: {len(relationships_df)}")
print(f"\nRelationship type distribution:")
print(relationships_df['relationship_type'].value_counts())

print(f"\nPredicate distribution:")
print(relationships_df['predicate'].value_counts())

# %%
# Preview
print("\nRelationship Edge List Preview:")
print(relationships_df.head(20))

# %% [markdown]
# ## Section 9: Validation - Sample Drug Analysis

# %%
print("Validation: Analyzing Sample Drugs")
print("=" * 80)

# Select 5 diverse drugs for validation
sample_drug_names = working_df['name'].head(5).tolist()

for drug_name in sample_drug_names:
    print(f"\n{'='*80}")
    print(f"DRUG: {drug_name}")
    print('='*80)
    
    # Get all relationships for this drug
    drug_rels = relationships_df[relationships_df['subject'] == drug_name]
    
    print(f"\nTotal relationships: {len(drug_rels)}")
    
    # By predicate
    print(f"\nRelationships by predicate:")
    for pred, count in drug_rels['predicate'].value_counts().items():
        print(f"  {pred:20s}: {count}")
    
    # Show samples
    print(f"\nSample relationships:")
    for idx, row in drug_rels.head(10).iterrows():
        print(f"  ({row['subject']}, {row['predicate']}, {row['object']})")

Creating Relationship Edge List:
✓ Total relationships: 103552

Relationship type distribution:
relationship_type
Drug-Drug_Interaction    103219
Drug-Condition              189
Drug-Adverse_Effect         144
Name: count, dtype: int64

Predicate distribution:
predicate
interacts_with    103219
treats               189
causes               144
Name: count, dtype: int64

Relationship Edge List Preview:
      subject       predicate                object      relationship_type  \
0   Lepirudin  interacts_with             Ceftezole  Drug-Drug_Interaction   
1   Lepirudin  interacts_with          Indomethacin  Drug-Drug_Interaction   
2   Lepirudin  interacts_with             Indalpine  Drug-Drug_Interaction   
3   Lepirudin  interacts_with              Warfarin  Drug-Drug_Interaction   
4   Lepirudin  interacts_with           Azacitidine  Drug-Drug_Interaction   
5   Lepirudin  interacts_with          Medrogestone  Drug-Drug_Interaction   
6   Lepirudin  interacts_with           Linsidomi

In [33]:
# ## Section 10: Generate Outputs

In [34]:
print("Generating Output Files:")
print("=" * 80)

# ============================================================================
# Output 1: Entity Catalog
# ============================================================================
entity_catalog_df.to_csv(ENTITY_CATALOG_PATH, index=False)
print(f"✓ Saved: {ENTITY_CATALOG_PATH}")
print(f"  Entities: {len(entity_catalog_df)}")
print(f"  Columns: {', '.join(entity_catalog_df.columns)}")

# ============================================================================
# Output 2: Relationship Edge List
# ============================================================================
relationships_df.to_csv(ENTITY_RELATIONSHIPS_PATH, index=False)
print(f"\n✓ Saved: {ENTITY_RELATIONSHIPS_PATH}")
print(f"  Relationships: {len(relationships_df)}")
print(f"  Columns: {', '.join(relationships_df.columns)}")

# ============================================================================
# Output 3: Extraction Report
# ============================================================================
extraction_report = {
    'extraction_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'input_drugs': int(len(working_df)),
    
    'entities': {
        'total': int(len(entity_catalog_df)),
        'drugs': int(len(drug_entities)),
        'adverse_effects': int(len(effect_entities)),
        'conditions': int(len(condition_entities))
    },
    
    'relationships': {
        'total': int(len(relationships_df)),
        'interacts_with': int(len(unique_interactions)),
        'causes': int(len(unique_toxicity)),
        'treats': int(len(unique_treats))
    },
    
    'extraction_quality': {
        'drugs_with_interactions': int(len([d for d in drug_entities if any(r['subject'] == d and r['predicate'] == 'interacts_with' for r in all_relationships)])),
        'drugs_with_toxicity': int(len([d for d in drug_entities if any(r['subject'] == d and r['predicate'] == 'causes' for r in all_relationships)])),
        'drugs_with_treats': int(len([d for d in drug_entities if any(r['subject'] == d and r['predicate'] == 'treats' for r in all_relationships)])),
        'avg_relationships_per_drug': float(len(relationships_df) / len(working_df))
    },
    
    'top_connected_drugs': {
        drug: int(count) 
        for drug, count in Counter([r['subject'] for r in all_relationships]).most_common(10)
    },
    
    'most_common_adverse_effects': {
        effect: int(count)
        for effect, count in toxicity_by_effect.most_common(10)
    },
    
    'most_treated_conditions': {
        condition: int(count)
        for condition, count in treats_by_condition.most_common(10)
    }
}

with open(EXTRACTION_REPORT_PATH, 'w') as f:
    json.dump(extraction_report, f, indent=2)

print(f"\n✓ Saved: {EXTRACTION_REPORT_PATH}")

Generating Output Files:
✓ Saved: ../Datasets/intermediate/entity_catalog.csv
  Entities: 149
  Columns: entity_id, entity_name, entity_type, source

✓ Saved: ../Datasets/intermediate/entity_relationships.csv
  Relationships: 103552
  Columns: subject, predicate, object, relationship_type, confidence, source

✓ Saved: ../Datasets/intermediate/entity_extraction_report.json


In [35]:
# ## Section 11: Final Summary

In [36]:
print("\n" + "=" * 80)
print("COMPONENT 2 COMPLETE - FINAL SUMMARY")
print("=" * 80)

print(f"\n✓ Input: {len(working_df)} selected drugs")

print(f"\n✓ Entities Extracted:")
print(f"  - Total: {len(entity_catalog_df)}")
print(f"  - Drugs: {len(drug_entities)}")
print(f"  - Adverse Effects: {len(effect_entities)}")
print(f"  - Conditions: {len(condition_entities)}")

print(f"\n✓ Relationships Extracted:")
print(f"  - Total: {len(relationships_df)}")
print(f"  - interacts_with: {len(unique_interactions)}")
print(f"  - causes: {len(unique_toxicity)}")
print(f"  - treats: {len(unique_treats)}")

print(f"\n✓ Extraction Coverage:")
drugs_with_any_rel = set([r['subject'] for r in all_relationships])
print(f"  - Drugs with relationships: {len(drugs_with_any_rel)}/{len(working_df)} ({len(drugs_with_any_rel)/len(working_df)*100:.1f}%)")
print(f"  - Avg relationships per drug: {len(relationships_df)/len(working_df):.1f}")

print(f"\n✓ Files Generated:")
print(f"  1. {ENTITY_CATALOG_PATH.name} ({len(entity_catalog_df)} entities)")
print(f"  2. {ENTITY_RELATIONSHIPS_PATH.name} ({len(relationships_df)} relationships)")
print(f"  3. {EXTRACTION_REPORT_PATH.name}")

print(f"\n✓ Next Steps:")
print(f"  1. Review extraction quality")
print(f"  2. Validate sample relationships")
print(f"  3. Check for any missing data")
print(f"  4. Proceed to Component 3: Concept & Domain Mapping")

print("\n" + "=" * 80)
print("END OF COMPONENT 2")
print("=" * 80)


COMPONENT 2 COMPLETE - FINAL SUMMARY

✓ Input: 120 selected drugs

✓ Entities Extracted:
  - Total: 149
  - Drugs: 95
  - Adverse Effects: 17
  - Conditions: 42

✓ Relationships Extracted:
  - Total: 103552
  - interacts_with: 103219
  - causes: 144
  - treats: 189

✓ Extraction Coverage:
  - Drugs with relationships: 100/120 (83.3%)
  - Avg relationships per drug: 862.9

✓ Files Generated:
  1. entity_catalog.csv (149 entities)
  2. entity_relationships.csv (103552 relationships)
  3. entity_extraction_report.json

✓ Next Steps:
  1. Review extraction quality
  2. Validate sample relationships
  3. Check for any missing data
  4. Proceed to Component 3: Concept & Domain Mapping

END OF COMPONENT 2
